# Self-Organizing Map (SOM) - TensorFlow / Keras

**Goal:** Cluster and visualize Iris features on a 2D grid.

This notebook favors clear, production-style structure: seeded runs,
explicit data preparation, small reusable modules, and compact
training loops that can be expanded for larger experiments.


## Architecture Notes

- **What it learns:** Competitive learning maps similar samples to nearby grid neurons.
- **Where it is used:** clustering, topology-preserving visualization, and exploratory analysis.
- **Why it works:** the architecture builds a useful bias into the computation, so the model does not need to rediscover that structure from data alone.
- **Output to expect:** classification models return class scores/probabilities, reconstruction models return reconstructed inputs, and generative models return new or denoised samples.


## Visual Intuition

Run this cell before or after training. It is lightweight and framework-independent, so it explains the network idea without requiring a long training run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
fig.suptitle("Self-Organizing Map: intuition, signal flow, and output", fontsize=13)
axes[0].axis("off")
layers = ['data', 'grid', 'clusters']
xs = np.linspace(0.1, 0.9, len(layers))
for xpos, label in zip(xs, layers):
    axes[0].scatter([xpos], [0.55], s=1200, color="#4C78A8", alpha=0.18, edgecolors="#4C78A8")
    axes[0].text(xpos, 0.55, label, ha="center", va="center", fontsize=9)
for a, b in zip(xs[:-1], xs[1:]):
    axes[0].annotate("", xy=(b - 0.045, 0.55), xytext=(a + 0.045, 0.55), arrowprops=dict(arrowstyle="->", lw=1.5))
axes[0].set_title("How data moves")
x = np.linspace(-3, 3, 160)
y = np.sin(2*x)
axes[1].plot(x, y, color="#F58518", lw=2)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_title("Toy behavior")
axes[1].grid(alpha=0.25)
values = np.array([.30,.20,.35,.15])
axes[2].bar(range(len(values)), values, color=["#54A24B", "#E45756", "#72B7B2", "#B279A2"][:len(values)])
axes[2].set_title("Typical output")
axes[2].set_xticks(range(len(values)))
axes[2].set_xticklabels(['n1', 'n2', 'n3', 'n4'])
axes[2].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(SEED)
print(f"TensorFlow: {tf.__version__}")
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

X = StandardScaler().fit_transform(load_iris().data).astype("float32")
data = tf.constant(X)
grid_h, grid_w = 8, 8
neurons = tf.Variable(tf.random.normal((grid_h, grid_w, X.shape[1])))
coords = tf.cast(tf.stack(tf.meshgrid(tf.range(grid_h), tf.range(grid_w), indexing="ij"), axis=-1), tf.float32)


In [ ]:
for step in range(600):
    sample = data[tf.random.uniform((), maxval=len(X), dtype=tf.int32)]
    distances = tf.reduce_sum(tf.square(neurons - sample), axis=-1)
    winner = tf.cast(tf.unravel_index(tf.argmin(tf.reshape(distances, [-1])), distances.shape), tf.float32)
    width = max(0.5, 3.0 * (1 - step / 600))
    lr = 0.3 * (1 - step / 600)
    influence = tf.exp(-tf.reduce_sum(tf.square(coords - winner), axis=-1) / (2 * width ** 2))
    neurons.assign_add(lr * influence[..., None] * (sample - neurons))

flattened = tf.reshape(neurons, (-1, X.shape[1]))
assignments = tf.argmin(tf.linalg.norm(data[:, None, :] - flattened[None, :, :], axis=-1), axis=1)
print("First 10 SOM neuron assignments:", assignments[:10].numpy().tolist())
